# Lab 13: Batch API for High-Throughput Processing (Gemini 3.1)

When you need to process thousands of requests that don't require an immediate response, the **Batch API** is the most efficient choice. It offers a **50% cost reduction** and higher throughput limits.

## Objectives
1. Prepare a request file in **JSONL** format.
2. Submit a **Batch Job** using the Gemini API.
3. Monitor job status and retrieve results.
4. Compare Batch vs. Online (Real-time) processing.

In [ ]:
import json
import os
import time

from dotenv import load_dotenv
from google import genai

load_dotenv()
client = genai.Client(api_key=os.getenv("GEMINI_API_KEY"))

print("Client initialized for Batch processing.")

## 1. Preparing the Batch Data (JSONL)

The Batch API requires a JSON Lines (.jsonl) file where each line is a valid request object.  
Each request must have a unique `key` and a `request` field containing a standard `GenerateContentRequest`.

In [ ]:
requests = [
    {
        "key": "req-1",
        "request": {
            "contents": [
                {"parts": [{"text": "Classify this sentiment: I love this product!"}]}
            ]
        }
    },
    {
        "key": "req-2",
        "request": {
            "contents": [{"parts": [{
                "text": "Classify this sentiment: It broke on the first day."
            }]}]
        }
    }
]

batch_filename = "batch_requests.jsonl"
with open(batch_filename, "w") as f:
    for req in requests:
        f.write(json.dumps(req) + "\n")

print(f"Prepared {len(requests)} requests in {batch_filename}.")

## 2. Uploading the File and Creating the Job

First, we upload the JSONL file to the File API, then we start the batch job.

In [ ]:
print("Uploading batch file...")
uploaded_file = client.files.upload(
    file=batch_filename,
    config={"display_name": batch_filename, "mime_type": "application/jsonl"}
)

print("Creating batch job...")
batch_job = client.batches.create(
    model="gemini-3.1-flash-lite-preview",
    src=uploaded_file.name
)

print(f"Batch Job Created! ID: {batch_job.name}")

## 3. Monitoring Job Progress

Batch jobs can take from a few minutes to several hours depending on the queue. We can poll the status.

In [ ]:
def wait_for_batch(job_id):
    while True:
        job = client.batches.get(name=job_id)
        status = job.state.name
        print(f"Current Status: {status}")

        if status in ["JOB_STATE_SUCCEEDED", "JOB_STATE_FAILED", "JOB_STATE_CANCELLED"]:
            return job

        time.sleep(10)  # Wait 10 seconds between checks

# Note: Monitoring job
completed_job = wait_for_batch(batch_job.name)


## 4. Retrieving and Parsing Results

Once the job is `SUCCEEDED`, the results are available in a new JSONL file stored in the File API. We download it using the API Key for authentication.

In [ ]:
if completed_job.state.name == "JOB_STATE_SUCCEEDED":
    output_file_name = completed_job.dest.file_name
    print(f"Results ready in file: {output_file_name}")

    # Download using the SDK's built-in file download method (returns bytes)
    print("Downloading results...")
    file_content = client.files.download(file=output_file_name)
    text_content = file_content.decode("utf-8")

    print("\n--- BATCH RESULTS ---\n")
    for line in text_content.strip().split("\n"):
        data = json.loads(line)
        key = data.get("key")
        try:
            answer = data["response"]["candidates"][0]["content"]["parts"][0]["text"]
            print(f"[{key}]: {answer.strip()}")
        except (KeyError, IndexError):
            print(
                f"[{key}]: Error parsing response — "
                f"{data.get('status', 'unknown error')}"
            )
else:
    print(
        f"Job failed with state: {completed_job.state.name}\n"
        f"Error: {completed_job.error}"
    )

## Summary

The Batch API is an essential tool for **cost-effective large-scale AI operations**:
1. **Scale**: Process millions of tokens without hitting real-time RPM limits.
2. **Cost**: Save 50% compared to standard pricing.
3. **Workflow**: Seamless automation from JSONL creation to results parsing.